# Lesson 01 - 人工智能代理簡介

歡迎來到 **AI代理初學者** 課程的第一課！

**AI代理** 是一個程式，它使用大型語言模型（LLM）作為推理引擎，並且能夠在真實世界中採取<em>行動</em>——呼叫API、查詢資料庫或執行程式碼——以代表使用者完成某項目標。

在這個筆記本中，您將建立您的第一個代理：一個推薦度假目的地的<strong>旅遊代理</strong>。在此過程中，您將學到如何：

1. 使用 **Microsoft Agent Framework** 連接到 Azure AI Foundry Agent 服務。
2. 給代理一個<strong>工具</strong>——一個它可以呼叫的純Python函式。
3. 運行代理並檢查其回應。
4. 逐字元串流代理的回應。


## 設定

在執行此筆記本之前，請確保您已經：

1. **擁有一個 Azure AI Foundry 專案**，並已部署了聊天模型（例如 `gpt-4o-mini`）。
2. **使用 Azure CLI 登入** — 在您的終端機中執行 `az login`。
3. **設定必要的環境變數：**
   - `AZURE_AI_PROJECT_ENDPOINT` — 您的 Azure AI Foundry 專案端點。
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — 您部署的模型名稱。

下面的程式格將會安裝您所需的 Python 套件。


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## 建立你的第一個 Agent

一個 agent 需要兩樣東西：

- <strong>指示</strong>，告訴它 <em>它是誰</em> 以及 <em>如何行事</em>（系統提示）。
- <strong>工具</strong> — 使用 `@tool` 裝飾的 Python 函數，agent 可以調用這些工具來取得資訊或執行動作。

以下我們定義一個簡單的工具，回傳熱門旅遊景點清單。當使用者要求旅行推薦時，agent 將使用此工具。


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## 串流回應

為了更互動的體驗，您可以<strong>串流</strong>代理的回應。代理會隨著產生文本片段逐步輸出，而不是等待整個回覆完成。在聊天介面中，這對於即時顯示輸出特別有用。


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Summary

在本課程中你學會了如何：

- <strong>創建一個提供者</strong>，透過 `AzureAIProjectAgentProvider` 連接到 Azure AI Foundry Agent Service。
- 使用 `@tool` 裝飾器 <strong>定義一個工具</strong>，讓代理可以調用你的 Python 函數。
- 使用用戶訊息 <strong>運行代理</strong> 並打印其回應。
- <strong>串流回應</strong> 以實現即時輸出。

在下一課中，我們將更深入探討代理框架，並學習如何賦予代理更強大的工具和多步推理能力。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們力求準確，但請注意，自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議尋求專業人工翻譯。我們不對因使用本翻譯而引起的任何誤解或曲解承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
